# Analyze biological replicate correlation neutralization assays

In [1]:
import os
import altair as alt
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error 
import numpy as np
import pandas as pd

# Basic color palette
color_palette = [
    '#345995', #blue
    '#03cea4', #teal
    '#ca1551', #red
    '#eac435', #yellow
    'grey'
               ]

In [2]:
# Find relative 'sera' directory in seqneut-pipeline results
seqneut_sera_dir = '../../results/sera'

resultsdir = 'results'
os.makedirs(resultsdir, exist_ok = True)

In [3]:

# replicate sera to concat titer data from
sera = [
    'UWMC_UWMC-1',
    'UWMC_UWMC-7',
    'UWMC_UWMC-9',
    'UWMC_UWMC-12',
    'UWMC_UWMC-18',
    'UWMC_UWMC-19',
    'UWMC_UWMC-20'
]

# initialize empty df
replicate_titers = pd.DataFrame()

# iterate through sera and concat titer data
for serum in sera:

    replicate_titers = pd.concat([replicate_titers, (pd.read_csv(os.path.join(seqneut_sera_dir, serum, 'titers_per_replicate.csv')))])

replicate_titers['fixed_replicate'] = replicate_titers['replicate'].str.replace(r'^([^-]+)-([^-]+)-([^-]+)$', r'\1_\2-\3', regex=True)

replicate_titers = replicate_titers.assign(plate = lambda x: x['fixed_replicate'].str.split('-').str[0],
                                barcode = lambda x: x['fixed_replicate'].str.split('-').str[1]
                               )

replicate_titers

,group,serum,virus,replicate,titer,titer_bound,titer_as,nt50,midpoint,top,bottom,slope,fixed_replicate,plate,barcode
0,UWMC,UWMC-1,A/Slovenia/49/2024_H3N2,plate1-AATCGCTGGCACCCGT,353.30,interpolated,midpoint,491.90,353.30,0.6613,0,3.4180,plate1-AATCGCTGGCACCCGT,plate1,AATCGCTGGCACCCGT
1,UWMC,UWMC-1,A/Slovenia/49/2024_H3N2,plate1-AATGAAACAATCGAAC,741.40,interpolated,midpoint,959.30,741.40,0.8741,0,1.1260,plate1-AATGAAACAATCGAAC,plate1,AATGAAACAATCGAAC
2,UWMC,UWMC-1,A/Slovenia/49/2024_H3N2,plate1-2-AATCGCTGGCACCCGT,349.00,interpolated,midpoint,523.00,349.00,0.6000,0,3.9780,plate1_2-AATCGCTGGCACCCGT,plate1_2,AATCGCTGGCACCCGT
3,UWMC,UWMC-1,A/Slovenia/49/2024_H3N2,plate1-2-AATGAAACAATCGAAC,318.60,interpolated,midpoint,542.10,318.60,0.6000,0,3.0280,plate1_2-AATGAAACAATCGAAC,plate1_2,AATGAAACAATCGAAC
4,UWMC,UWMC-1,A/Netherlands/01502/2025_H3N2,plate1-GAGGGGTAGAGATACG,1163.00,interpolated,midpoint,1163.00,1163.00,1.0000,0,0.9320,plate1-GAGGGGTAGAGATACG,plate1,GAGGGGTAGAGATACG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
577,UWMC,UWMC-20,A/Singapore/INFIMH-16-0019/2016_H3N2,plate2-TACCAATGTCATTTGA,56.31,interpolated,midpoint,64.67,56.31,0.6254,0,9.9850,plate2-TACCAATGTCATTTGA,plate2,TACCAATGTCATTTGA
578,UWMC,UWMC-20,A/Singapore/INFIMH-16-0019/2016_H3N2,plate2-TACTAGCAATAAAATC,1408.00,interpolated,midpoint,1492.00,1408.00,0.9773,0,0.8000,plate2-TACTAGCAATAAAATC,plate2,TACTAGCAATAAAATC
579,UWMC,UWMC-20,A/Singapore/INFIMH-16-0019/2016_H3N2,plate2-2-CGTACAGTGTAATCGA,398.40,interpolated,midpoint,398.40,398.40,1.0000,0,0.8043,plate2_2-CGTACAGTGTAATCGA,plate2_2,CGTACAGTGTAATCGA
580,UWMC,UWMC-20,A/Singapore/INFIMH-16-0019/2016_H3N2,plate2-2-TACCAATGTCATTTGA,237.80,interpolated,midpoint,255.30,237.80,0.7451,0,10.0000,plate2_2-TACCAATGTCATTTGA,plate2_2,TACCAATGTCATTTGA


In [4]:
replicate_titers_pivot = (replicate_titers
     .pivot(index = ['serum', 'virus', 'barcode'],
                           columns = 'plate',
                           values = 'titer'
                          )
     .reset_index()
     .assign(log_fold_change_2729 = lambda x: np.log(x['plate1']/x['plate1_2']),
             log_fold_change_2830 = lambda x: np.log(x['plate2']/x['plate2_2']))
    )

print(replicate_titers_pivot.serum.unique())
replicate_titers_pivot
# replicate_titers_pivot.query('serum =="PENN23_y1981_s053_d28"')

['UWMC-1' 'UWMC-12' 'UWMC-18' 'UWMC-19' 'UWMC-20' 'UWMC-7' 'UWMC-9']


plate,serum,virus,barcode,plate1,plate1_2,plate2,plate2_2,log_fold_change_2729,log_fold_change_2830
0,UWMC-1,A/Amapa/021563-IEC/2024_H3N2,AGTTGGGGTCTCCCTT,709.0,472.4,NaN,NaN,0.406029,NaN
1,UWMC-1,A/Amapa/021563-IEC/2024_H3N2,TTGTATCAGTCGCGCC,938.6,NaN,NaN,NaN,NaN,NaN
2,UWMC-1,A/Badajoz/18680568/2025_H3N2,GCCTTTGCGCGCAGTC,2770.0,1652.0,NaN,NaN,0.516861,NaN
3,UWMC-1,A/Badajoz/18680568/2025_H3N2,TTTCACAGAACCTATC,625.0,3171.0,NaN,NaN,-1.624051,NaN
4,UWMC-1,A/Bangkok/P176/2025_H1N1,AGATCCACCCTATAGT,307.9,290.6,NaN,NaN,0.057827,NaN
...,...,...,...,...,...,...,...,...,...
2049,UWMC-9,A/Wisconsin/NIRC-IS-1028/2024_H3N2,TCAATCGGGGGCTAAA,1157.0,NaN,NaN,NaN,NaN,NaN
2050,UWMC-9,A/Wisconsin/NIRC-IS-1111/2025_H1N1,AGATCCCAGGTCCTTT,178.8,162.5,NaN,NaN,0.095590,NaN
2051,UWMC-9,A/Wisconsin/NIRC-IS-1111/2025_H1N1,TTAATGTAGCCGCTCC,234.2,419.6,NaN,NaN,-0.583126,NaN
2052,UWMC-9,A/Zacapa/FLU-012/2025_H1N1,AGTTTTTATAACTTGC,158.2,123.3,NaN,NaN,0.249240,NaN


## Calculate R2 correlation and RMSD

In [5]:
# write values to dictionary
# initialize empty dictionary

corr_values = {}
corr_values_string = {}

# iterate through sera and calculate R2 across experimental replicate titers

for s in replicate_titers_pivot.serum.unique():
    s_number = (int(s.split('-')[1]))
    if s_number <= 11:
        col1 = 'plate1'
        col2 = 'plate1_2'
    elif s_number > 11:
        col1 = 'plate2'
        col2 = 'plate2_2'


    # reduce dataframe to relevant sera, plates
    df = replicate_titers_pivot.query(f'serum == "{s}"')[[col1, col2]].dropna(axis=0)

    # log-transform titers
    titer1 = np.log2(df[col1])
    titer2 = np.log2(df[col2])
    
    # reshape to 2D arrays for sklearn
    titer1 = titer1.values.reshape(-1, 1)
    titer2 = titer2.values.reshape(-1, 1)
    
    # fit model
    model = LinearRegression()
    model.fit(titer1, titer2)
    
    #calculate R-squared of regression model
    r_squared = model.score(titer1, titer2)
    
    # calcualte mse
    rmse = root_mean_squared_error(titer1, titer2)

    corr_values[s] = r_squared
    corr_values_string[s] = s + ', r2=' + str(r_squared)[0:5]

print('saving dictionary of sera matched with R2...')
print(corr_values)

saving dictionary of sera matched with R2...
{'UWMC-1': 0.6633370398699959, 'UWMC-12': 0.5307733553096012, 'UWMC-18': 0.25626142420721454, 'UWMC-19': 0.36635965253204317, 'UWMC-20': 0.5437577141841405, 'UWMC-7': 0.30834274103813786, 'UWMC-9': 0.6751374666658871}


## Correlations on viral strain level

In [6]:
replicate_NT50_pivot = (replicate_titers
     .pivot(index = ['serum', 'virus', 'barcode'],
                           columns = 'plate',
                           values = 'nt50'
                          )
     .reset_index()
     .assign(log_fold_change_2729 = lambda x: np.log(x['plate1']/x['plate1_2']),
             log_fold_change_2830 = lambda x: np.log(x['plate2']/x['plate2_2']))
    )


# intialize empty list for median values
median_titer_ls = []

# get medium titers per virus per sera
for v in replicate_NT50_pivot.virus.unique():
    for s in replicate_NT50_pivot.serum.unique():

        df = replicate_NT50_pivot.query(f'virus == "{v}"').query(f'serum == "{s}"')
       
        s_number = (int(s.split('-')[1]))
        if s_number <= 11:
            df = df.dropna(subset=['plate1', 'plate1_2'])
            plate1_median = (df.plate1.median())
            plate1_2_median = (df.plate1_2.median())
            plate2_median = np.nan
            plate2_2_median = np.nan
        elif s_number > 11:
            df = df.dropna(subset=['plate2', 'plate2_2'])
            plate1_median = np.nan
            plate1_2_median = np.nan
            plate2_median = (df.plate2.median())
            plate2_2_median = (df.plate2_2.median())

        median_titer_ls.append([
            s, v,               
            plate1_median,
            plate1_2_median, 
            plate2_median,
            plate2_2_median,
        ])

# make df
median_titer_df = pd.DataFrame(median_titer_ls, columns = ['serum', 'virus', 
                                                           'plate1_median', 'plate1_2_median', 
                                                           'plate2_median', 'plate2_2_median',])
# merge df with pivot
replicate_NT50_median_merge = replicate_NT50_pivot.merge(median_titer_df, how = 'left', on = ['serum', 'virus'])

replicate_NT50_median_merge

,serum,virus,barcode,plate1,plate1_2,plate2,plate2_2,log_fold_change_2729,log_fold_change_2830,plate1_median,plate1_2_median,plate2_median,plate2_2_median
0,UWMC-1,A/Amapa/021563-IEC/2024_H3N2,AGTTGGGGTCTCCCTT,780.6,955.0,NaN,NaN,-0.201648,NaN,780.60,955.0,NaN,NaN
1,UWMC-1,A/Amapa/021563-IEC/2024_H3N2,TTGTATCAGTCGCGCC,949.9,NaN,NaN,NaN,NaN,NaN,780.60,955.0,NaN,NaN
2,UWMC-1,A/Badajoz/18680568/2025_H3N2,GCCTTTGCGCGCAGTC,2770.0,3322.0,NaN,NaN,-0.181720,NaN,1742.65,8471.0,NaN,NaN
3,UWMC-1,A/Badajoz/18680568/2025_H3N2,TTTCACAGAACCTATC,715.3,13620.0,NaN,NaN,-2.946593,NaN,1742.65,8471.0,NaN,NaN
4,UWMC-1,A/Bangkok/P176/2025_H1N1,AGATCCACCCTATAGT,319.8,496.7,NaN,NaN,-0.440290,NaN,242.30,493.8,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2049,UWMC-9,A/Wisconsin/NIRC-IS-1028/2024_H3N2,TCAATCGGGGGCTAAA,1157.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2050,UWMC-9,A/Wisconsin/NIRC-IS-1111/2025_H1N1,AGATCCCAGGTCCTTT,194.7,228.8,NaN,NaN,-0.161388,NaN,226.95,418.2,NaN,NaN
2051,UWMC-9,A/Wisconsin/NIRC-IS-1111/2025_H1N1,TTAATGTAGCCGCTCC,259.2,607.6,NaN,NaN,-0.851917,NaN,226.95,418.2,NaN,NaN
2052,UWMC-9,A/Zacapa/FLU-012/2025_H1N1,AGTTTTTATAACTTGC,164.2,130.3,NaN,NaN,0.231246,NaN,186.50,164.3,NaN,NaN


In [7]:
UWMC_20_titers = (replicate_NT50_median_merge.query('serum == "UWMC-20"')
                  .dropna(subset=['plate2_median', 'plate2_2_median'])
                  [['serum', 'virus', 'plate2_median', 'plate2_2_median']]
                  .drop_duplicates()
                 )
UWMC_20_titers


# log-transform titers
titer1 = np.log2(UWMC_20_titers['plate2_median'])
titer2 = np.log2(UWMC_20_titers['plate2_2_median'])

# reshape to 2D arrays for sklearn
titer1 = titer1.values.reshape(-1, 1)
titer2 = titer2.values.reshape(-1, 1)

# fit model
model = LinearRegression()
model.fit(titer1, titer2)

#calculate R-squared of regression model
r_squared = model.score(titer1, titer2)
print(r_squared)
# calculate R
r = np.sqrt(r_squared) * np.sign(model.coef_[0][0])
print(r)
UWMC_20_titers

0.7756737466966545
0.8807234223617846


,serum,virus,plate2_median,plate2_2_median
1176,UWMC-20,A/Amapa/021563-IEC/2024_H3N2,862.450,890.050
1178,UWMC-20,A/Badajoz/18680568/2025_H3N2,421.250,539.700
1180,UWMC-20,A/Bangkok/P176/2025_H1N1,103.610,153.150
1182,UWMC-20,A/Brisbane/02/2018_H1N1,1163.000,591.600
1185,UWMC-20,A/BurkinaFaso/3131/2023_H3N2,294.450,447.500
...,...,...,...,...
1458,UWMC-20,A/Wisconsin/588/2019_H1N1,200.800,110.500
1461,UWMC-20,A/Wisconsin/67/2022_H1N1,70.365,90.785
1464,UWMC-20,A/Wisconsin/NIRC-IS-1028/2024_H3N2,718.900,785.650
1466,UWMC-20,A/Wisconsin/NIRC-IS-1111/2025_H1N1,70.415,105.345


In [8]:
# for median only
temp_df = replicate_NT50_median_merge[['serum', 'virus', 
                                       'plate1_median', 'plate1_2_median', 
                                       'plate2_median', 'plate2_2_median',]].drop_duplicates()

# write values to dictionary
# initialize empty dictionary
virus_corr_values = {}
virus_corr_values_string = {}

# iterate through sera and calculate R2 across experimental replicate titers

for s in temp_df.serum.unique():
    s_number = (int(s.split('-')[1]))
    if s_number <= 11:
        col1 = 'plate1_median'
        col2 = 'plate1_2_median'
    elif s_number > 11:
        col1 = 'plate2_median'
        col2 = 'plate2_2_median'
        
    # reduce dataframe to relevant sera, plates
    df = temp_df.query(f'serum == "{s}"')[[col1, col2]].dropna().reset_index(drop = True)

    # log-transform titers
    titer1 = np.log2(df[col1])
    titer2 = np.log2(df[col2])
    
    # reshape to 2D arrays for sklearn
    titer1 = titer1.values.reshape(-1, 1)
    titer2 = titer2.values.reshape(-1, 1)
    
    # fit model
    model = LinearRegression()
    model.fit(titer1, titer2)
    
    #calculate R-squared of regression model
    r_squared = model.score(titer1, titer2)

    # calculate R
    r = np.sqrt(r_squared) * np.sign(model.coef_[0][0])
    
    # calcualte mse
    rmse = root_mean_squared_error(titer1, titer2)
    
    virus_corr_values[s] = str(r)[0:5]
    virus_corr_values_string[s] = s + ', r=' + str(r)[0:5]
    # virus_corr_values[s] = r_squared
    # virus_corr_values_string[s] = s + ', r2=' + str(r_squared)[0:5]

print('saving dictionary of sera matched with R...')
print(virus_corr_values)

saving dictionary of sera matched with R...
{'UWMC-1': '0.777', 'UWMC-12': '0.805', 'UWMC-18': '0.637', 'UWMC-19': '0.756', 'UWMC-20': '0.880', 'UWMC-7': '0.517', 'UWMC-9': '0.720'}


In [11]:
# configure color scheme
fill = True
opacity = 0.5
stroke = 'black'
strokeWidth = 1.8
markSize = 120
RtextLocation = 60

sort_list = ['UWMC-1', 'UWMC-7', 'UWMC-9']

def sort_dict_by_list(d, order):
    # First get values for the keys that are in the order list
    ordered_vals = [d[k] for k in order if k in d]
    # Then add values for keys not in the order list
    remaining_vals = [v for k, v in d.items() if k not in order]
    return ordered_vals + remaining_vals

sorted_sera = (sort_dict_by_list(virus_corr_values_string, sort_list))

color = alt.Color('serum', sort = sorted_sera, legend=None)

titleFontSize = 16
labelFontSize = 16

width = 180
height = width
_range = [60, 30000]


# add serum, R2 column
df = (temp_df
      .assign(
          serum_label = lambda x: x.serum,
          serum_number = lambda x: x.serum.str.split('-').str[1].astype(int)
      )
      .replace({'serum_label': virus_corr_values_string})
      .sort_values(by='serum_number')
      .reset_index(drop=True)
       )

plots = []

for sera in df.serum.unique():
    r_string = virus_corr_values[sera]
    serum_df = df.query(f'serum == "{sera}"')
    s_number = (int(sera.split('-')[1]))
    if s_number <= 11:
        col1 = 'plate1_median'
        col2 = 'plate1_2_median'
    elif s_number > 11:
        col1 = 'plate2_median'
        col2 = 'plate2_2_median'

    barcode_scatter = (
        alt.Chart(serum_df, width = width, height = height, title=alt.Title(sera, fontSize=titleFontSize, fontStyle='light'))
        .mark_circle(size=markSize, opacity=opacity, stroke=stroke, strokeWidth = strokeWidth, filled=fill)
        .encode(
            alt.X(col1, 
                  title = ['neutralization titer', 'replicate 1'],
                  scale = alt.Scale(nice=False, padding=6, type="log", domain=_range),
                  axis = alt.Axis(grid=False, titleFontSize=titleFontSize, titleFontStyle='light', labelFontSize=labelFontSize)
                 ),
            alt.Y(col2,
                  title = ['neutralization titer', 'replicate 2'],
                  scale = alt.Scale(nice=False, padding=6, type="log", domain=_range),
                  axis = alt.Axis(grid=False, titleFontSize=titleFontSize, titleFontStyle='light', labelFontSize=labelFontSize)
                 ),
            # color = color,
            tooltip=['serum', 'virus', col1, col2])
    )

    # Text annotation
    text = alt.Chart(pd.DataFrame({
            'label': ['r='+r_string]
        })).mark_text(
            align='right',
            baseline='top',
            dx=RtextLocation+20, dy=RtextLocation,
            fontSize=labelFontSize,
            # fontStyle='italic',
            color='black'
        ).encode(
            text='label:N'
        )

    # Combine and append
    plots.append(barcode_scatter + text)

    # break

concat = (alt.concat(*plots, 
                    columns = 3, 
                    spacing = 10
                   )
    )

# Save
outfile = os.path.join(resultsdir, 'per_strain_replicate_correlations.pdf')
concat.save(outfile, dpi = 600)
concat


alt.ConcatChart(...)